In [11]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import featuretools as ft
import matplotlib.pyplot as plt
import seaborn as sns
import re
from woodwork import logical_types
from featuretools.primitives import AggregationPrimitive
from collections import Counter
from featuretools.utils.gen_utils import Library
from woodwork.column_schema import ColumnSchema

In [31]:
# Load datasets and limit to the first 1000 rows
app_train = pd.read_csv('home-credit-default-risk/application_train.csv').sort_values('SK_ID_CURR').reset_index(drop=True).loc[:4999, :]
app_test = pd.read_csv('home-credit-default-risk/application_test.csv').sort_values('SK_ID_CURR').reset_index(drop=True).loc[:4999, :]
bureau = pd.read_csv('home-credit-default-risk/bureau.csv').sort_values(['SK_ID_CURR', 'SK_ID_BUREAU']).reset_index(drop=True).loc[:4999, :]
bureau_balance = pd.read_csv('home-credit-default-risk/bureau_balance.csv').sort_values('SK_ID_BUREAU').reset_index(drop=True).loc[:4999, :]
cash = pd.read_csv('home-credit-default-risk/POS_CASH_balance.csv').sort_values(['SK_ID_CURR', 'SK_ID_PREV']).reset_index(drop=True).loc[:4999, :]
credit = pd.read_csv('home-credit-default-risk/credit_card_balance.csv').sort_values(['SK_ID_CURR', 'SK_ID_PREV']).reset_index(drop=True).loc[:4999, :]
previous = pd.read_csv('home-credit-default-risk/previous_application.csv').sort_values(['SK_ID_CURR', 'SK_ID_PREV']).reset_index(drop=True).loc[:4999, :]
installments = pd.read_csv('home-credit-default-risk/installments_payments.csv').sort_values(['SK_ID_CURR', 'SK_ID_PREV']).reset_index(drop=True).loc[:4999, :]

# Correctly identify special logical types

In [32]:
app_types = {}

# Iterate through the columns and record the Boolean columns
for col in app_train:
    # If column is a number with only two values, encode it as a Boolean
    if (app_train[col].dtype != 'object') and (len(app_train[col].unique()) <= 2):
        app_types[col] = logical_types.Boolean

print('Number of boolean variables: ', len(app_types))

app_types['REGION_RATING_CLIENT'] = logical_types.Categorical
app_types['REGION_RATING_CLIENT_W_CITY'] = logical_types.Categorical

prev_types = {}

# Iterate through the columns and record the Boolean columns
for col in previous:
    # If column is a number with only two values, encode it as a Boolean
    if (previous[col].dtype != 'object') and (len(previous[col].unique()) <= 2):
        prev_types[col] = logical_types.Boolean

print('Number of boolean variables: ', len(prev_types))

def replace_day_outliers(df):
    """Replace 365243 with np.nan in any columns with DAYS"""
    for col in df.columns:
        if "DAYS" in col:
            df[col] = df[col].replace({365243: np.nan})

    return df

# Replace all the day outliers
app_train = replace_day_outliers(app_train)
app_test = replace_day_outliers(app_test)
bureau = replace_day_outliers(bureau)
bureau_balance = replace_day_outliers(bureau_balance)
credit = replace_day_outliers(credit)
cash = replace_day_outliers(cash)
previous = replace_day_outliers(previous)
installments = replace_day_outliers(installments)

Number of boolean variables:  33
Number of boolean variables:  1


# Create Time-Related Variables

In [33]:
# Establish a starting date for all applications at Home Credit
start_date = pd.Timestamp("2016-01-01")

# Convert to timedelta in days
for col in ['DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE']:
    bureau[col] = pd.to_timedelta(bureau[col], 'D')
    
# Create the date columns
bureau['bureau_credit_application_date'] = start_date + bureau['DAYS_CREDIT']
bureau['bureau_credit_end_date'] = start_date + bureau['DAYS_CREDIT_ENDDATE']
bureau['bureau_credit_close_date'] = start_date + bureau['DAYS_ENDDATE_FACT']
bureau['bureau_credit_update_date'] = start_date + bureau['DAYS_CREDIT_UPDATE']

# Drop the time offset columns
bureau = bureau.drop(columns = ['DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE'])

# Convert to timedelta
bureau_balance['MONTHS_BALANCE'] = pd.to_timedelta(bureau_balance['MONTHS_BALANCE'] * 30, 'D')

# Make a date column
bureau_balance['bureau_balance_date'] = start_date + bureau_balance['MONTHS_BALANCE']
bureau_balance = bureau_balance.drop(columns = ['MONTHS_BALANCE'])

# Convert to timedeltas in days
for col in ['DAYS_DECISION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']:
    previous[col] = pd.to_timedelta(previous[col], 'D')
    
# Make date columns
previous['previous_decision_date'] = start_date + previous['DAYS_DECISION']
previous['previous_drawing_date'] = start_date + previous['DAYS_FIRST_DRAWING']
previous['previous_first_due_date'] = start_date + previous['DAYS_FIRST_DUE']
previous['previous_last_duefirst_date'] = start_date + previous['DAYS_LAST_DUE_1ST_VERSION']
previous['previous_last_due_date'] = start_date + previous['DAYS_LAST_DUE']
previous['previous_termination_date'] = start_date + previous['DAYS_TERMINATION']

# Drop the time offset columns
previous = previous.drop(columns = ['DAYS_DECISION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION'])


# Convert to timedelta objects
credit['MONTHS_BALANCE'] = pd.to_timedelta(credit['MONTHS_BALANCE']* 30, 'D')
cash['MONTHS_BALANCE'] = pd.to_timedelta(cash['MONTHS_BALANCE']* 30, 'D')

# Make a date column
credit['credit_balance_date'] = start_date + credit['MONTHS_BALANCE']
credit = credit.drop(columns = ['MONTHS_BALANCE'])

# Make a date column
cash['cash_balance_date'] = start_date + cash['MONTHS_BALANCE']
cash = cash.drop(columns = ['MONTHS_BALANCE'])

# Convert to time delta object
installments['DAYS_INSTALMENT'] = pd.to_timedelta(installments['DAYS_INSTALMENT'], 'D')
installments['DAYS_ENTRY_PAYMENT'] = pd.to_timedelta(installments['DAYS_ENTRY_PAYMENT'], 'D')

# Create time column and drop
installments['installments_due_date'] = start_date + installments['DAYS_INSTALMENT']
installments = installments.drop(columns = ['DAYS_INSTALMENT'])

installments['installments_paid_date'] = start_date + installments['DAYS_ENTRY_PAYMENT']
installments = installments.drop(columns = ['DAYS_ENTRY_PAYMENT'])



# Add dataframes to entityset

In [34]:
es = ft.EntitySet(id="client_data")
es = es.add_dataframe(dataframe_name="app_train",
                      dataframe=app_train,
                      index="SK_ID_CURR",
                      logical_types=app_types)

es = es.add_dataframe(dataframe_name="bureau",
                      dataframe=bureau,
                      index="SK_ID_BUREAU",
                      time_index='bureau_credit_application_date')

es = es.add_dataframe(dataframe_name="previous",
                      dataframe=previous,
                      index="SK_ID_PREV",
                      logical_types=prev_types)

#make index
es = es.add_dataframe(dataframe_name = 'bureau_balance',
                      dataframe = bureau_balance,
                      make_index = True,
                      index = 'bureaubalance_index',
                      time_index = 'bureau_balance_date')

es = es.add_dataframe(dataframe_name = 'cash',
                      dataframe = cash,
                      make_index = True,
                      index = 'cash_index',
                      time_index = 'cash_balance_date')

es = es.add_dataframe(dataframe_name = 'installments',
                      dataframe = installments,
                      make_index = True,
                      index = 'installments_index',
                      time_index = 'installments_paid_date')

es = es.add_dataframe(dataframe_name = 'credit',
                      dataframe = credit,
                      make_index = True,
                      index = 'credit_index',
                      time_index = 'credit_balance_date')
print(es)

c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer forma

Entityset: client_data
  DataFrames:
    app_train [Rows: 5000, Columns: 122]
    bureau [Rows: 5000, Columns: 17]
    previous [Rows: 5000, Columns: 37]
    bureau_balance [Rows: 5000, Columns: 4]
    cash [Rows: 5000, Columns: 9]
    installments [Rows: 5000, Columns: 9]
    credit [Rows: 5000, Columns: 24]
  Relationships:
    No relationships


c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(


# Relationships

In [35]:
#add a relationship between app_train and bureau
es.add_relationship(parent_dataframe_name="app_train",
    parent_column_name="SK_ID_CURR",
    child_dataframe_name="bureau",
    child_column_name="SK_ID_CURR")

#add a relationship between bureau and bureau_balance
es.add_relationship(parent_dataframe_name="bureau",
    parent_column_name="SK_ID_BUREAU",
    child_dataframe_name="bureau_balance",
    child_column_name="SK_ID_BUREAU")

#add a relationship between current app and previous app
es.add_relationship(parent_dataframe_name="app_train",
    parent_column_name="SK_ID_CURR",
    child_dataframe_name="previous",
    child_column_name="SK_ID_CURR")

#relationships between previous apps and cash, installments, and credit
es.add_relationship(parent_dataframe_name="previous",
    parent_column_name="SK_ID_PREV",
    child_dataframe_name="cash",
    child_column_name="SK_ID_PREV")

es.add_relationship(parent_dataframe_name="previous",
    parent_column_name="SK_ID_PREV",
    child_dataframe_name="installments",
    child_column_name="SK_ID_PREV")

es.add_relationship(parent_dataframe_name="previous",
    parent_column_name="SK_ID_PREV",
    child_dataframe_name="credit",
    child_column_name="SK_ID_PREV")

print(es)
#es.plot()

Entityset: client_data
  DataFrames:
    app_train [Rows: 5000, Columns: 122]
    bureau [Rows: 5000, Columns: 17]
    previous [Rows: 5000, Columns: 37]
    bureau_balance [Rows: 5000, Columns: 4]
    cash [Rows: 5000, Columns: 9]
    installments [Rows: 5000, Columns: 9]
    credit [Rows: 5000, Columns: 24]
  Relationships:
    bureau.SK_ID_CURR -> app_train.SK_ID_CURR
    bureau_balance.SK_ID_BUREAU -> bureau.SK_ID_BUREAU
    previous.SK_ID_CURR -> app_train.SK_ID_CURR
    cash.SK_ID_PREV -> previous.SK_ID_PREV
    installments.SK_ID_PREV -> previous.SK_ID_PREV
    credit.SK_ID_PREV -> previous.SK_ID_PREV


# Interesting Values

In [36]:
interesting_values = {'NAME_CONTRACT_STATUS':['Approved', 'Refused', 'Canceled']}
es.add_interesting_values(dataframe_name="previous", values=interesting_values)

# Seed Features

In [37]:
# Late Payment seed feature
late_payment = ft.Feature(es['installments'].ww['installments_due_date']) < ft.Feature(es['installments'].ww['installments_paid_date'])

# Rename the feature
late_payment = late_payment.rename("late_payment")

# Create a feed representing whether the loan is past due
past_due = ft.Feature(es['bureau_balance'].ww['STATUS']).isin(['1', '2', '3', '4', '5'])
past_due = past_due.rename("past_due")

# Custom Aggregators

In [38]:
class NormalizedModeCount(AggregationPrimitive):
    name = "NormalizedModeCount"
    input_types = [ColumnSchema(semantic_tags={'category'})]
    return_type = ColumnSchema(semantic_tags={'numeric'})
    compatibility = [Library.PANDAS, Library.DASK, Library.SPARK]

    def get_function(self):
        def normalized_mode_count(x):
            if x.mode().shape[0] == 0:
                return np.nan
            counts = dict(Counter(x.values))
            mode = x.mode().iloc[0]
            return counts[mode] / np.sum(list(counts.values()))
        return normalized_mode_count


class LongestSeq(AggregationPrimitive):
    name = "LongestSeq"
    input_types = [ColumnSchema(semantic_tags={'category'})]
    # It's fine to have a Categorical return type for an aggregation primitive
    return_type = ColumnSchema(semantic_tags={'category'})
    compatibility = [Library.PANDAS, Library.DASK, Library.SPARK]

    def get_function(self):
        def longest_repetition(x):
            x = x.dropna()
            if x.shape[0] < 1:
                return None

            longest_element = current_element = None
            longest_repeats = current_repeats = 0

            for element in x:
                if current_element == element:
                    current_repeats += 1
                else:
                    current_element = element
                    current_repeats = 1
                if current_repeats > longest_repeats:
                    longest_repeats = current_repeats
                    longest_element = current_element

            return longest_element
        return longest_repetition
    

class MostRecent(AggregationPrimitive):
    name = "MostRecent"
    input_types = [
        ColumnSchema(semantic_tags={'category'}),
        ColumnSchema(logical_type=logical_types.Datetime)
    ]
    return_type = ColumnSchema(semantic_tags={'category'})
    compatibility = [Library.PANDAS, Library.DASK, Library.SPARK]

    def get_function(self):
        def most_recent(y, x):
            df = pd.DataFrame({"x": x, "y": y}).dropna()
                    
            if df.shape[0] < 1:
                return np.nan

            df = df.sort_values('x', ascending=False).reset_index()

            return df.iloc[0]['y']
        return most_recent

# Run Final Deep Feature Search

In [39]:
# Run and create the features

categorical_columns = {}
for df in es.dataframes:
    df_name = df.ww.name
    df_categorical_cols = df.ww.select('categorical').columns.to_list()
    categorical_columns[df_name] = df_categorical_cols
print(categorical_columns)

primitive_options = {
    "cum_mean": {
        "ignore_columns": categorical_columns
    },
    "cum_sum": {
        "ignore_columns": categorical_columns
    },
    "diff": {
        "ignore_columns": categorical_columns
    },
    NormalizedModeCount: {
        "ignore_columns": categorical_columns
    }
}

print(primitive_options)
feature_matrix_test, feature_names_test = ft.dfs(entityset = es, target_dataframe_name = 'app_train',
                                                   agg_primitives = ['mean', 'max', 'min', 'trend', 'mode', 'count', 
                                                                     'sum', 'percent_true', NormalizedModeCount, MostRecent, LongestSeq],
                                                   trans_primitives = ['diff', 'cum_sum', 'cum_mean', 'percentile'], 
                                                   where_primitives = ['mean', 'sum'],
                                                   seed_features = [late_payment, past_due],
                                                   max_depth = 2, features_only = False, verbose = True,
                                                   chunk_size = len(app_test),
                                                   primitive_options=primitive_options)

{'app_train': ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE'], 'bureau': ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE'], 'previous': ['NAME_CONTRACT_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION'], 'bureau_balance': ['STATUS'], 'cash': ['NAME_CONTRACT_STATUS'], 'installments': [], 'credit': ['NAME_CONTRACT_STATUS']}
{'cum_mean': {'ignore_columns': {'app_train': ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_

c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\featuretools\synthesis\dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  agg_primitives: ['NormalizedModeCount']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)


Elapsed: 07:11 | Progress:  95%|█████████▌

c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer forma

Elapsed: 07:24 | Progress:  95%|█████████▌

c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\gamer\miniconda3\envs\Atari_Gym\lib\site-packages\woodwork\type_sys\utils.py:40: UserWarning: Could not infer forma

Elapsed: 07:33 | Progress: 100%|██████████


In [40]:
from featuretools import selection

# Remove low information features
feature_matrix2 = selection.remove_low_information_features(feature_matrix_test)
print('Removed %d features from training features'  % (feature_matrix_test.shape[1] - feature_matrix2.shape[1]))

Removed 1007 features from training features


In [41]:
feature_matrix_test.to_csv('feature_matrix.csv')
feature_matrix2.to_csv('feature_matrix_trimmed.csv')

: 